In [4]:
import random
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI


In [18]:
import os


model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",   # or "gemini-1.5-pro", "gemini-1.5-flash"
    google_api_key=os.getenv("GOOGLE_API_KEY"),  # or set GOOGLE_API_KEY env var
)


In [19]:
# agent = create_agent(model, tools=[word_count], checkpointer=InMemorySaver())



# bishal = {"configurable": {"thread_id": "student-bishal"}}

# def say(config, text):
#     r = agent.invoke({"messages": [{"role": "user", "content": text}]}, config)
#     print("agent:", r["messages"][-1].content)

# say(bishal, "My name is Bishal.")     # stored on this thread
# say(bishal, "What is my name?")       # -> recalled from memory



In [20]:
me = {"configurable": {"thread_id": "another_thread_test"}} 

In [21]:


import sqlite3

DB_PATH = "notes.db"

def init_db():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS notes (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            text TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    conn.close()

In [22]:



@tool
def save_note(text: str) -> str:
    """Save a fact the user wants remembered for later into the database i.e, sqlite"""
    conn = sqlite3.connect(DB_PATH)
    conn.execute("INSERT INTO notes (text) VALUES (?)", (text,))
    conn.commit()
    conn.close()
    return "saved"


init_db()

@tool
def list_notes() -> str:
    """List everything the user has asked to remember from the database"""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.execute(
        "SELECT text FROM notes ORDER BY id ASC"
    )
    notes = [row[0] for row in cursor.fetchall()]
    conn.close()

    return "; ".join(notes) or "(nothing saved yet)"

@tool
def delete_note(note_id: int) -> str:
    """Delete a specific saved note by its ID."""
    conn = sqlite3.connect(DB_PATH)

    cursor = conn.execute(
        "DELETE FROM notes WHERE id = ?",
        (note_id,)
    )

    conn.commit()
    deleted = cursor.rowcount
    conn.close()

    if deleted:
        return f"note {note_id} deleted"
    return f"note {note_id} not found"


In [41]:

TOOLS = [ save_note, list_notes, delete_note, roll_dice,decide_for_me]
assistant = create_agent(model, tools=TOOLS, checkpointer=InMemorySaver())
me = {"configurable": {"thread_id": "another_thread_test"}} 

def say(config, text):
    r = assistant.invoke({"messages": [{"role": "user", "content": text}]}, config)
    print("agent:", r["messages"][-1].content)

    
#say(me, "save this fact 'ram is a good boy'")
say(me, "delete the note that has id 6 ")



agent: [{'type': 'text', 'text': "I couldn't find a note with ID 6. Would you like me to list your saved notes so you can find the correct ID?", 'extras': {'signature': 'EpoECpcEARFNMg+FD3IVRNrGFYCS9bG51ccuUBbIqQuSwUravfYG/8jmXr6U/5OEixd0DFR3jBAxhNRPyQUUNZJ4faGlKjKxyrGw4SKHkSFZLZ8Klp1f3L1RueQobu/huxYybVFixUOimSPHcx45n10xE4ponCS1edeHUzEfykJFNLeJ2ROkit/D0lHvAJzR/RauOxLZJrx/EiPLEh2jzkJwkQMju9bi3icye+3eRh7PUMPDxLO0PBpaolUgyKB6yije+WbtqIFXUro31JSdIQqH8deGo3wn3I8cfzYH/DLWDHv7AxfWMiDbcTwYxGuIqAqJYhSMxUvu5ALbyY5AqqZZPqGFm1ZoYjvNrpq3Wl4EaUb2GVrHpEzrqjFJc8XpkP40MFD3AZuAeJjwLnrEZFtrLw0l6f/f0J06W8T2h5lzBl4DwyIvIJIPYFPx1QfqmLKoD86447yPkUZFftLP1X6PTvO/9A9jFG8i/ihlS8SJMYm4ousfqVoW+HajFUmc+iti5VSzizjuR0gJY7wcinRZOaOCiacUKRc5fiPlXezJ/Cc5GgYuEYNLd8QEiMp2anrLYAMGNgS+NnOddnqiUPOHEFOaCQyp7Aef+u3LawyR5u2OHZ1T0CQObBC3fyFuOb/A9IFe4G6dTuCztIQReAhxnlrUn6rDOfBjoe6lWQJs+0JNajuUiLcvybQyWF4AV/r+T8qtN0gtwXZWzA=='}}]


In [42]:
def list_notes() -> str:
    """List everything the user has asked to remember."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.execute(
        "SELECT text FROM notes ORDER BY id ASC"
    )
    notes = [row[0] for row in cursor.fetchall()]
    conn.close()

    return "; ".join(notes) or "(nothing saved yet)"


res = list_notes()

In [43]:
res

'(nothing saved yet)'

In [44]:
# def save_note(text: str) -> str:
#     """Save a fact the user wants remembered for later."""
#     conn = sqlite3.connect(DB_PATH)
#     conn.execute("INSERT INTO notes (text) VALUES (?)", (text,))
#     conn.commit()
#     conn.close()
#     return "saved"


# res = save_note("ram is a good boy")


In [45]:

def delete_all_notes() -> str:
    """Delete everything saved in the notes table."""
    conn = sqlite3.connect(DB_PATH)
    conn.execute("DELETE FROM notes")
    conn.commit()
    conn.close()

    return "all notes deleted"


delete_all_notes()

'all notes deleted'

In [46]:
def list_notes() -> str:
    """List everything the user has asked to remember."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.execute(
        "SELECT * FROM notes "
    )
    notes = [row[0] for row in cursor.fetchall()]
    conn.close()

    return (notes) or "(nothing saved yet)"


res = list_notes()

In [47]:
res

'(nothing saved yet)'

In [48]:
@tool
def roll_dice(sides: int, times: int) -> str:
    """Roll a dice with the given number of sides, a given number of times."""
    rolls = [random.randint(1, sides) for _ in range(times)]
    return f"rolled {rolls}, total = {sum(rolls)}"

In [49]:
say(me, "roll 3 six-sided dice")

agent: [{'type': 'text', 'text': 'You rolled a 2, 3, and 6, for a total of 11!', 'extras': {'signature': 'EsYBCsMBARFNMg+kGlHPlgAmKa1s/YdiqSwdr0duxtE5ffwKxZmGjcFuEj/lew5wihB2mqKQq/XnEncmnYmt2ddkFXxdhHoeNMxHFG7p+cPNuqlDSuBLo4jP5XIOE6IKm/lsXv0yrSVW+7OQP2bDZFbfGG4lefmfQOuLjajuNIYKNQVyB51F7IpIvyg9/q1GxtxmRS5gmVUjVoG3AjJBKbceF2lBbtNIaxT2KPXYbp1DHb1ycoedj1XQnd+NDxqe0gPW6LBJFSkK'}}]


In [62]:
@tool
def decide_for_me(options: list[str]) -> str:
    """Pick one option at random when the user can't decide between several things."""
    if not options:
        return "give me some options first"
    return f"I choose: {random.choice(options)}"

In [63]:
say(me, "I can't decide between momo, thukpa and chowmein")

GoogleRateLimitError: Error calling model 'gemini-3.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 13.779432109s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', '